<a href="https://colab.research.google.com/github/smerarawal/IIT-BHU-Proj/blob/main/kimore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import os
import glob

def extract_features(data):
    features = {}
    joints = ['rHip', 'rKnee', 'rAnkle', 'lHip', 'lKnee', 'lAnkle']
    for joint in joints:
        idx = {'rHip': 12, 'rKnee': 13, 'rAnkle': 14, 'lHip': 15, 'lKnee': 16, 'lAnkle': 17}[joint]
        coords = data.iloc[:, idx*4:idx*4+3].values
        features[f'{joint}_ROM'] = np.ptp(coords, axis=0).mean()
        features[f'{joint}_peak_angle'] = coords.max()
        features[f'{joint}_start_angle'] = coords[0].mean()
        features[f'{joint}_end_angle'] = coords[-1].mean()
        features[f'{joint}_max_velocity'] = 0.0
        features[f'{joint}_mean_velocity'] = 0.0
    for col in [f'{j}_{metric}' for j in joints for metric in ['angle_mean', 'angle_std', 'mean_vel_desc', 'mean_vel_asc', 'vel_skewness', 'vel_kurtosis', 'EC_ratio', 'mean_jerk', 'peak_jerk', 'smoothness', 'hesitations']]:
        features[col] = 0.0
    return features

base_dir = '/content/drive/MyDrive/rehab_dataset/kimore_dataset'
files = glob.glob(os.path.join(base_dir, 'Train_X*.csv'))
processed_data = []

for file in files:
    df = pd.read_csv(file, header=None)
    feat = extract_features(df)
    feat['rep_id'] = os.path.basename(file)
    feat['exercise'] = 'KIMORE_Protocol'
    feat['valgus_angle'] = 0.0
    feat['target_label'] = 1
    feat['subject_id'] = 'kimore_sub'
    processed_data.append(feat)

df_kimore = pd.DataFrame(processed_data)
df_kimore.to_csv("kimore_features.csv", index=False)

In [4]:
import os
from google.colab import drive
drive.mount('/content/drive')
print(os.listdir('/content/drive/MyDrive/rehab_dataset/kimore_dataset/'))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/rehab_dataset/kimore_dataset/'

In [5]:
!ls -R /content/drive/MyDrive/ | grep -i "rehab"

ucophyrehab_final
ucophyrehab2_data (1).jsonl


In [7]:
import pandas as pd
import numpy as np
import glob
import os

# 1. Ensure path points to local Colab storage
base_dir = '/content/'
output_file = 'kimore_features_final_last.csv'

# 2. Block-aware extraction function
def extract_kimore_features(file_path):
    df = pd.read_csv(file_path, header=None)

    # Joint map: adjust these indices if your CSV columns are different
    joint_map = {'rHip': 12, 'rKnee': 13, 'rAnkle': 14, 'lHip': 15, 'lKnee': 16, 'lAnkle': 17}

    features = {}
    for joint, idx in joint_map.items():
        # Slice the 4-column block: (x, y, z, confidence)
        coords = df.iloc[:, idx*4 : idx*4+3].values

        # Extract kinematics
        features[f'{joint}_ROM'] = np.ptp(coords, axis=0).mean()
        features[f'{joint}_peak_angle'] = coords.max()
        features[f'{joint}_start_angle'] = coords[0].mean()
        features[f'{joint}_end_angle'] = coords[-1].mean()
        features[f'{joint}_max_velocity'] = 0.0
        features[f'{joint}_mean_velocity'] = 0.0

    return features

# 3. Batch process and save
files = glob.glob(os.path.join(base_dir, 'Train_X*.csv'))
processed_data = []

for file in files:
    try:
        feat = extract_kimore_features(file)
        feat['rep_id'] = os.path.basename(file)
        feat['exercise'] = 'KIMORE_Protocol'
        feat['target_label'] = 1
        processed_data.append(feat)
    except Exception as e:
        print(f"Skipping {file}: {e}")

# 4. Save
df_kimore = pd.DataFrame(processed_data)
df_kimore.to_csv(output_file, index=False)
print(f"Extraction successful: {len(df_kimore)} files processed.")

Extraction successful: 1 files processed.


In [8]:
import pandas as pd
import numpy as np
import glob
import os

base_dir = '/content/'
output_file = 'kimore_features_final1.csv'

def extract_kimore_features(df_subset):
    joint_map = {'rHip': 12, 'rKnee': 13, 'rAnkle': 14, 'lHip': 15, 'lKnee': 16, 'lAnkle': 17}
    features = {}
    for joint, idx in joint_map.items():
        coords = df_subset.iloc[:, idx*4 : idx*4+3].values
        features[f'{joint}_ROM'] = np.ptp(coords, axis=0).mean()
        features[f'{joint}_peak_angle'] = coords.max()
        features[f'{joint}_start_angle'] = coords[0].mean()
        features[f'{joint}_end_angle'] = coords[-1].mean()
        features[f'{joint}_max_velocity'] = 0.0
        features[f'{joint}_mean_velocity'] = 0.0
    for col in [f'{j}_{metric}' for j in joint_map.keys() for metric in ['angle_mean', 'angle_std', 'mean_vel_desc', 'mean_vel_asc', 'vel_skewness', 'vel_kurtosis', 'EC_ratio', 'mean_jerk', 'peak_jerk', 'smoothness', 'hesitations']]:
        features[col] = 0.0
    return features

files = glob.glob(os.path.join(base_dir, 'Train_X*.csv'))
all_frames = []

for file in files:
    df = pd.read_csv(file, header=None)
    for frame_idx in range(len(df)):
        feat = extract_kimore_features(df.iloc[[frame_idx]])
        feat['rep_id'] = f"{os.path.basename(file)}_frame_{frame_idx}"
        feat['exercise'] = 'KIMORE_Protocol'
        feat['target_label'] = 1
        all_frames.append(feat)

df_kimore = pd.DataFrame(all_frames)
df_kimore.to_csv(output_file, index=False)

In [9]:
def extract_kimore_features(df_subset, prev_df_subset=None):
    joint_map = {'rHip': 12, 'rKnee': 13, 'rAnkle': 14, 'lHip': 15, 'lKnee': 16, 'lAnkle': 17}
    features = {}

    for joint, idx in joint_map.items():
        coords = df_subset.iloc[:, idx*4 : idx*4+3].values.flatten()
        features[f'{joint}_ROM'] = coords.mean() # Simplified for single frame

        # Calculate Velocity if previous frame exists
        if prev_df_subset is not None:
            prev_coords = prev_df_subset.iloc[:, idx*4 : idx*4+3].values.flatten()
            velocity = np.linalg.norm(coords - prev_coords)
            features[f'{joint}_mean_velocity'] = velocity
        else:
            features[f'{joint}_mean_velocity'] = 0.0

    return features

In [10]:
df = pd.read_csv('kimore_features_final.csv')
# Fill gaps and remove extreme noise
df = df.replace([np.inf, -np.inf], np.nan).fillna(0)
df.to_csv('kimore_features_cleaned.csv', index=False)

In [11]:
all_frames = []

for file in files:
    df = pd.read_csv(file, header=None)

    # Process every single frame in the file
    for i in range(len(df)):
        # Extract features for THIS specific row
        feat = extract_kimore_features(df.iloc[[i]])

        # Add metadata for the frame
        feat['rep_id'] = f"{os.path.basename(file)}_frame_{i}"
        feat['exercise'] = 'KIMORE_Protocol'
        feat['target_label'] = 1 # Replace with your logic to read Train_Y

        all_frames.append(feat)

# This will result in a DataFrame with one row per frame
df_kimore = pd.DataFrame(all_frames)
df_kimore.to_csv("kimore_features_final2.csv", index=False)

In [15]:
# ╔═════════════════════════════════════════════════╕╗
# ║  KIMORE Feature Extractor — Colab Ready                                ║
# ║  1. from google.colab import drive; drive.mount('/content/drive')       ║
# ║  2. Set DATA_DIR to your folder path                                    ║
# ║  3. Edit PAIRS if your filenames differ                                 ║
# ╖═════════════════════════════════════════════════╕╝

# !pip install scipy openpyxl -q

import numpy as np
import pandas as pd
from scipy.signal import savgol_filter
from scipy.stats  import skew, kurtosis
from scipy.ndimage import label as nd_label
import os, warnings
import re # Import re module
warnings.filterwarnings('ignore')

# ── Constants ────────────────────────────────────────────────────
FPS            = 100
FRAMES_PER_REP = 100
SG_WIN         = 11
SG_POLY        = 3
HESIT_THRESH   = 2.0   # °/s

# ── KIMORE exercise name map ──────────────────────────────────
# File number → exercise name (from KIMORE paper, Capecci et al. 2019)
# Standard KIMORE has exercises 1-5. File numbers >5 are extended dataset.
EXERCISE_NAMES = {
    '1': 'lateral_trunk_bending',
    '2': 'lifting',
    '3': 'sit_to_stand',
    '4': 'one_leg_balance',
    '5': 'squat',
    # Extended KIMORE — add names if you know them, otherwise kept as-is
    '6': 'exercise_6',
    '7': 'exercise_7',
    '8': 'exercise_8',
}

# ── Kinect v2 joint indices (x, y, z, conf = 4 cols per joint) ───────────────
JOINT_IDX = {
    'SpineBase': 0,
    'HipLeft'  :12, 'KneeLeft'  :13, 'AnkleLeft' :14, 'FootLeft'  :15,
    'HipRight' :16, 'KneeRight' :17, 'AnkleRight':18, 'FootRight' :19,
}

# ── Angle computation ─────────────────────────────

def get_xyz(frames, jname):
    i = JOINT_IDX[jname]
    return frames[:, i*4 : i*4+3].astype(np.float64)

def joint_angle(a, b, c):
    """Angle in degrees at joint b between segments b→a and b→c."""
    v1 = a - b;  v2 = c - b
    cos_a = np.sum(v1*v2, axis=1) / (
        np.linalg.norm(v1,axis=1) * np.linalg.norm(v2,axis=1) + 1e-8)
    return np.degrees(np.arccos(np.clip(cos_a, -1.0, 1.0)))

def compute_angles(frames):
    hl=get_xyz(frames,'HipLeft');   kl=get_xyz(frames,'KneeLeft')
    al=get_xyz(frames,'AnkleLeft'); fl=get_xyz(frames,'FootLeft')
    hr=get_xyz(frames,'HipRight');  kr=get_xyz(frames,'KneeRight')
    ar=get_xyz(frames,'AnkleRight');fr=get_xyz(frames,'FootRight')
    sp=get_xyz(frames,'SpineBase')
    return {
        'rHip'  : joint_angle(kr, hr, sp),
        'rKnee' : joint_angle(hr, kr, ar),
        'rAnkle': joint_angle(kr, ar, fr),
        'lHip'  : joint_angle(kl, hl, sp),
        'lKnee' : joint_angle(hl, kl, al),
        'lAnkle': joint_angle(kl, al, fl),
    }

# ── Feature helpers ───────────────────────────────────────

def smooth(arr):
    win = SG_WIN if len(arr) >= SG_WIN else (len(arr) | 1)
    win = max(win, SG_POLY+2 if (SG_POLY+2)%2==1 else SG_POLY+3)
    return savgol_filter(arr, window_length=win, polyorder=SG_POLY)

def velocity(arr):
    return np.gradient(arr, 1.0 / FPS)

def smoothness_cv(arr):
    """
    1 - CV(|velocity|). Returns [0,1] where 1 = smooth.
    Uses velocity CV instead of spectral methods (SPARC/LDJ) because those
    need longer signals; CV works correctly on 100-frame reps.
    """
    v = np.abs(velocity(arr))
    cv = np.std(v) / (np.mean(v) + 1e-8)
    return float(np.clip(1.0 - cv / 3.0, 0.0, 1.0))

def hesitation_count(vel):
    labeled, n = nd_label(np.abs(vel) < HESIT_THRESH)
    return max(0, n)

def ec_ratio(arr):
    """Frames to peak / frames from peak. 1.0 = symmetric timing."""
    peak_idx = int(np.argmin(arr))
    n = len(arr)
    if peak_idx == 0 or peak_idx == n-1:
        return 1.0
    return float(peak_idx / max(n - peak_idx, 1))

def joint_features(angle_raw, prefix):
    s   = smooth(angle_raw)
    vel = velocity(s)
    jk  = np.abs(np.gradient(vel, 1.0/FPS))

    peak_idx      = int(np.argmin(s))
    mean_vel_desc = float(np.mean(np.abs(vel[:peak_idx+1]))) if peak_idx > 0 else 0.0
    mean_vel_asc  = float(np.mean(np.abs(vel[peak_idx:])))   if peak_idx < len(vel)-1 else 0.0

    return {
        f'{prefix}_ROM'          : float(np.max(s) - np.min(s)),
        f'{prefix}_peak_angle'   : float(np.min(s)),
        f'{prefix}_start_angle'  : float(s[0]),
        f'{prefix}_end_angle'    : float(s[-1]),
        f'{prefix}_max_velocity' : float(np.max(np.abs(vel))),
        f'{prefix}_mean_velocity': float(np.mean(np.abs(vel))),
        f'{prefix}_angle_mean'   : float(np.mean(s)),
        f'{prefix}_angle_std'    : float(np.std(s)),
        f'{prefix}_mean_vel_desc': mean_vel_desc,
        f'{prefix}_mean_vel_asc' : mean_vel_asc,
        f'{prefix}_vel_skewness' : float(skew(vel)),
        f'{prefix}_vel_kurtosis' : float(kurtosis(vel)),
        f'{prefix}_EC_ratio'     : ec_ratio(s),
        f'{prefix}_mean_jerk'    : float(np.mean(jk)),
        f'{prefix}_peak_jerk'    : float(np.max(jk)),
        f'{prefix}_smoothness'   : smoothness_cv(s),
        f'{prefix}_hesitations'  : hesitation_count(vel),
    }

# ── File identification helper ───────────────────────────

def identify_file(path):
    """Return ('skeleton', n_reps) or ('scores', n_reps) for any KIMORE CSV."""
    df = pd.read_csv(path, header=None)
    if df.shape[1] == 100:
        return 'skeleton', df.shape[0] // FRAMES_PER_REP
    elif df.shape[1] == 1:
        return 'scores', df.shape[0]
    else:
        raise ValueError(f'Unexpected shape {df.shape} in {path}')

def parse_file_number(filename):
    """Extract exercise number from KIMORE filename like Train_X__1___2_.csv"""
    parts = os.path.basename(filename).replace('.csv','').split('__')
    return parts[1].strip('_')

# ── Process one pair ───────────────────────────

def process_pair(skel_path, score_path, exercise_num, subject_id=None):
    """
    exercise_num : string like '1', '5', '7' — used to look up exercise name
    """
    skel   = pd.read_csv(skel_path,  header=None).values.astype(np.float64)
    scores = pd.read_csv(score_path, header=None).values.ravel().astype(np.float64)
    n_reps = len(scores)

    if skel.shape[0] != n_reps * FRAMES_PER_REP:
        raise ValueError(
            f'Rep count mismatch: skeleton={skel.shape[0]//FRAMES_PER_REP}, '
            f'scores={n_reps} in {os.path.basename(skel_path)}')

    if subject_id is None:
        # Original: subject_id = os.path.basename(skel_path).split('__')[2].replace('.csv','').strip('_')
        # New parsing for filenames like 'Train_X (1) (2).csv'
        base_filename = os.path.basename(skel_path)
        match = re.search(r'\((\d+)\)\.csv$', base_filename)
        if match:
            subject_id = match.group(1)
        else:
            subject_id = 'unknown' # Fallback for unexpected formats

    exercise_name = EXERCISE_NAMES.get(str(exercise_num), f'exercise_{exercise_num}')

    rows = []
    for rep_i in range(n_reps):
        frames = skel[rep_i * FRAMES_PER_REP : (rep_i+1) * FRAMES_PER_REP]
        angles = compute_angles(frames)
        feats  = {}
        for jname in ['rHip','rKnee','rAnkle','lHip','lKnee','lAnkle']:
            feats.update(joint_features(angles[jname], jname))
        feats['rep_id']          = f'E{exercise_num}_S{subject_id}_R{rep_i:04d}'
        feats['exercise']        = exercise_name        # ← human-readable name
        feats['exercise_num']    = f'E{exercise_num}'   # ← keep numeric ID too
        feats['subject_id']      = subject_id
        feats['target_label']    = float(scores[rep_i])
        rows.append(feats)

    return pd.DataFrame(rows)

def reorder_columns(df):
    joint_order = ['rHip','rKnee','rAnkle','lHip','lKnee','lAnkle']
    basic    = ['ROM','peak_angle','start_angle','end_angle',
                'max_velocity','mean_velocity']
    dynamics = ['angle_mean','angle_std','mean_vel_desc','mean_vel_asc',
                'vel_skewness','vel_kurtosis','EC_ratio',
                'mean_jerk','peak_jerk','smoothness','hesitations']
    feature_cols = ([f'{j}_{b}' for j in joint_order for b in basic] +
                    [f'{j}_{d}' for j in joint_order for d in dynamics])
    meta_cols = ['rep_id','exercise','exercise_num','subject_id','target_label']
    return df[feature_cols + meta_cols]

# ══════════════════════════════════════════════════════════════════════════
# ── CONFIGURE HERE ──────────────────────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════

DATA_DIR = '/content/'   # ← your Drive path

# How to identify pairs:
#   pd.read_csv(f, header=None).shape  →  (N*100, 100) = skeleton
#                                          (N, 1)       = scores
#
# Format: (skeleton_file, scores_file, exercise_number)
# exercise_number is used to look up the name in EXERCISE_NAMES above.
#
# From the uploaded files (verified by rep count matching):
#   skeleton               scores                  exercise_num
#   Train_X__1___2_.csv    Train_Y__2___1_.csv     1  (lateral trunk bending, 473 reps)
#   Train_Y__4___1_.csv    Train_X__5___1_.csv     4  (one leg balance, 530 reps)
#   Train_X__7___1_.csv    Train_Y__8___1_.csv     7  (576 reps, rater 1)
#   Train_X__7___1_.csv    Train_Y__8___2_.csv     7  (576 reps, rater 2)
#   Train_X__7___1_.csv    Train_Y__8___3_.csv     7  (576 reps, rater 3)
#
# For E7 with 3 raters — options:
#   (a) keep all 3 rows (done below) and average later
#   (b) replace the 3 E7 entries with a single averaged-score call (see note below)

PAIRS = [
    ('Train_X (1) (2).csv', 'Train_Y (2) (1).csv', '1'),
    ('Train_Y (4) (1).csv', 'Train_X (5) (1).csv', '4'), # Corrected: swapped skeleton and scores filenames
    ('Train_X (7) (1).csv', 'Train_Y (8) (1).csv', '7'),
    ('Train_X (7) (1).csv', 'Train_Y (8) (2).csv', '7'),
    ('Train_X (7) (1).csv', 'Train_Y (8) (3).csv', '7'),
]

# ══════════════════════════════════════════════════════════════════════════
# ── RUN ──────────────────────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════

all_dfs = []
rater_counts = {}   # track E7 rater index for rep_id uniqueness

for skel_f, score_f, ex_num in PAIRS:
    skel_path  = os.path.join(DATA_DIR, skel_f)
    score_path = os.path.join(DATA_DIR, score_f)
    ex_name    = EXERCISE_NAMES.get(ex_num, f'exercise_{ex_num}')

    # give each E7 rater a unique subject_id so rep_ids don't collide
    rater_counts[ex_num] = rater_counts.get(ex_num, 0) + 1
    subj_id = f'r{rater_counts[ex_num]}' if rater_counts[ex_num] > 1 else None

    print(f'Processing E{ex_num} ({ex_name}) ...')
    df = process_pair(skel_path, score_path, ex_num, subject_id=subj_id)
    print(f'  {len(df)} reps  |  score {df.target_label.min():.1f}–{df.target_label.max():.1f}  '
          f'|  rKnee ROM mean {df.rKnee_ROM.mean():.1f}°')
    all_dfs.append(df)

combined = reorder_columns(pd.concat(all_dfs, ignore_index=True))

# ── Optional: average the 3 E7 rater scores into one row per rep ─────────────
# Uncomment this block if you want deduplicated E7 (recommended for training)
#
# e7_mask = combined['exercise_num'] == 'E7'
# e7      = combined[e7_mask].copy()
# non_e7  = combined[~e7_mask]
# e7['rater'] = e7.groupby('rep_id').cumcount()
# e7_avg = e7.copy()
# e7_avg['target_label'] = e7.groupby(
#     e7['rep_id'].str.replace('_r[23]','',regex=True))['target_label'].transform('mean')
# e7_avg = e7_avg[e7_avg['rater']==0].drop(columns='rater')
# combined = pd.concat([non_e7, e7_avg], ignore_index=True)
# print(f'After averaging E7 raters: {len(combined)} reps')

# ── Save ──────────────────────────────────────────────────────────────────────────
OUT_PATH = os.path.join(DATA_DIR, 'kimore_features_last.csv')
combined.to_csv(OUT_PATH, index=False)

print(f'\nSaved {len(combined)} reps × {len(combined.columns)} cols → {OUT_PATH}')
print(f'NaN count: {combined.isna().sum().sum()}')
print('\nROM summary by exercise (why trunk bending has low knee ROM):')
print(combined.groupby('exercise')[['rKnee_ROM','lKnee_ROM','rHip_ROM']].mean().round(1).to_string())
print('\nTarget label stats:')
print(combined.groupby('exercise')['target_label'].describe().round(2).to_string())

Processing E1 (lateral_trunk_bending) ...
  473 reps  |  score 14.0–50.0  |  rKnee ROM mean 10.1°
Processing E4 (one_leg_balance) ...
  530 reps  |  score 10.0–50.0  |  rKnee ROM mean 17.5°
Processing E7 (exercise_7) ...
  576 reps  |  score 10.0–50.0  |  rKnee ROM mean 14.6°
Processing E7 (exercise_7) ...
  576 reps  |  score 10.0–50.0  |  rKnee ROM mean 14.6°
Processing E7 (exercise_7) ...
  576 reps  |  score 10.0–50.0  |  rKnee ROM mean 14.6°

Saved 2731 reps × 107 cols → /content/kimore_features_last.csv
NaN count: 0

ROM summary by exercise (why trunk bending has low knee ROM):
                       rKnee_ROM  lKnee_ROM  rHip_ROM
exercise                                             
exercise_7                  14.6       15.3      14.6
lateral_trunk_bending       10.1       11.5       7.0
one_leg_balance             17.5       18.0      25.2

Target label stats:
                        count   mean    std   min    25%   50%    75%   max
exercise                                  